# Week3_ex2 — Series capacitor compensation of the 300 km line

Reproduces Example 5.10 from Glover, Overbye & Sarma, *Power System Analysis and Design* (7th ed.): the 765 kV line of Example 5.2 with a series capacitor at each end (30 % compensation in total), solved with both capacitors bypassed and in service. The one-line shows the case as stored, with both capacitors bypassed.

![PowerWorld one-line](Images/Week3_ex2_result_image.png)

In [1]:
# 1. Case data and per-unit conversion
import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.topology as top

S_BASE = 500.0                    # MVA, Simulator Options > Power Flow Solution > General
V_KV = 765.0
F_HZ = 60.0
Z_BASE = V_KV**2 / S_BASE
X_LINE, B_LINE = 0.08279, 1.66050   # Branches Input, 3-4 Line, R = 0
X_CAP = -0.01240                    # Branches Input, 1-3 and 4-2 Series Cap, one at each end
X_BYPASS = 1e-5                     # PowerWorld swaps a bypassed capacitor for a "low impedance branch" without
                                    # stating the value; 1e-5 pu is the round value that reproduces the stored case
MVA_TOL = 0.5                       # PowerWorld MVA convergence tolerance

X_OHM = X_LINE * Z_BASE
C_NF = B_LINE / Z_BASE / (2 * np.pi * F_HZ) * 1e9

pd.DataFrame({
    "Quantity": ["System base", "Line 3-4", "Series cap 1-3 and 4-2", "Compensation", "Cap bypassed", "Cap in service"],
    "PowerWorld": ["500 MVA", "X = 0.08279 pu, B = 1.66050 pu", "X = -0.01240 pu each",
                   "2 x 0.01240 / 0.08279", "Bypassed: YES (Status stays Closed)", "Bypassed: NO"],
    "pandapower": ["sn_mva=500", f"x_ohm_per_km={X_OHM:.2f}, c_nf_per_km={C_NF:.1f}",
                   f"create_impedance, xft_pu={X_CAP}", f"{-2 * X_CAP / X_LINE:.1%}",
                   f"impedance, xft_pu={X_BYPASS}", "impedance in service"],
})

,Quantity,PowerWorld,pandapower
0,System base,500 MVA,sn_mva=500
1,Line 3-4,"X = 0.08279 pu, B = 1.66050 pu","x_ohm_per_km=96.90, c_nf_per_km=3763.2"
2,Series cap 1-3 and 4-2,X = -0.01240 pu each,"create_impedance, xft_pu=-0.0124"
3,Compensation,2 x 0.01240 / 0.08279,30.0%
4,Cap bypassed,Bypassed: YES (Status stays Closed),"impedance, xft_pu=1e-05"
5,Cap in service,Bypassed: NO,impedance in service


In [2]:
# 2. Build and solve: bypassed at 2200 MW, in service at 2200 MW and 3000 MW
# Buses are created in PowerWorld number order, so pandapower index = PowerWorld number - 1
BUSES = ["Bus A", "Bus B", "Three", "Four"]


def build_case(p_mw, caps_in, q_mvar=0.0):
    net = pp.create_empty_network(f_hz=F_HZ, sn_mva=S_BASE)
    b = [pp.create_bus(net, vn_kv=V_KV, name=n) for n in BUSES]
    pp.create_ext_grid(net, b[0], vm_pu=1.0, name="Slack")
    pp.create_line_from_parameters(net, b[2], b[3], length_km=1.0, r_ohm_per_km=0.0, x_ohm_per_km=X_OHM,
                                   c_nf_per_km=C_NF, max_i_ka=10.0, name="300 km line")
    for f, t in [(b[0], b[2]), (b[3], b[1])]:
        # A bypassed capacitor is shorted out, not disconnected: current keeps flowing through the bypass path
        pp.create_impedance(net, f, t, rft_pu=0.0, xft_pu=X_CAP if caps_in else X_BYPASS, sn_mva=S_BASE,
                            name="Series cap" if caps_in else "Bypass")
    pp.create_load(net, b[1], p_mw=p_mw, q_mvar=q_mvar, name="Load")
    pp.runpp(net, numba=False)
    return net


def results(net):
    out = {}
    for i, name in enumerate(BUSES[1:], start=1):
        out[f"{name} voltage [kV]"] = net.res_bus.vm_pu.iat[i] * V_KV
        out[f"{name} angle [deg]"] = net.res_bus.va_degree.iat[i]
    out["Slack P [MW]"] = net.res_ext_grid.p_mw.iat[0]
    out["Slack Q [Mvar]"] = net.res_ext_grid.q_mvar.iat[0]
    return out


STATES = {"Bypassed, 2200 MW": (2200.0, False), "In service, 2200 MW": (2200.0, True),
          "In service, 3000 MW": (3000.0, True)}
nets = {s: build_case(p, caps) for s, (p, caps) in STATES.items()}

pd.DataFrame({s: results(n) for s, n in nets.items()}).round(3)

,"Bypassed, 2200 MW","In service, 2200 MW","In service, 3000 MW"
Bus B voltage [kV],765.011,777.176,749.024
Bus B angle [deg],-21.368,-14.920,-21.364
Three voltage [kV],765.000,761.275,773.370
Three angle [deg],-0.003,3.143,4.220
Four voltage [kV],765.011,778.262,751.277
Four angle [deg],-21.365,-17.946,-25.802
Slack P [MW],2200.000,2200.000,3000.000
Slack Q [Mvar],-0.163,-256.689,330.636


In [3]:
# 3. Compare against PowerWorld
# Same rule as for the uncompensated line: the tolerance is the larger of half the last displayed digit and
# how far each quantity moves when +/-0.5 MW or +/-0.5 Mvar of residual is pushed into the load.
def residual_band(p_mw, caps_in):
    base = pd.Series(results(build_case(p_mw, caps_in)))
    moved = [pd.Series(results(build_case(p_mw + dp, caps_in, dq))) - base
             for dp, dq in [(MVA_TOL, 0.0), (-MVA_TOL, 0.0), (0.0, MVA_TOL), (0.0, -MVA_TOL)]]
    return pd.concat(moved, axis=1).abs().max(axis=1)


def compare(rows):
    """Build a PowerWorld vs pandapower table and fail loudly on any miss."""
    df = pd.DataFrame(rows, columns=["Case", "Quantity", "PowerWorld", "pandapower", "Tolerance", "Source"])
    df["Abs error"] = (df["pandapower"] - df["PowerWorld"]).abs()
    bad = df[df["Abs error"] > df["Tolerance"]]
    assert bad.empty, bad
    return df.round({"pandapower": 3, "Tolerance": 4, "Abs error": 4})


# PowerWorld values read once from PowerWorld Simulator (Run Mode, Model Explorer > Network > Buses),
# solved again after every change; order follows results(): Bus B, Three, Four, then the slack
PW = {
    "Bypassed, 2200 MW": [765.010, -21.37, 765.000, -0.00, 765.010, -21.37, 2200.00, -0.16],
    "In service, 2200 MW": [777.175, -14.92, 761.275, 3.14, 778.261, -17.95, 2200.00, -256.69],
    "In service, 3000 MW": [749.022, -21.36, 773.370, 4.22, 751.275, -25.80, 3000.01, 330.64],
}
DISPLAY = [5e-4, 5e-3] * 3 + [MVA_TOL, MVA_TOL]

rows = []
for state, pw_values in PW.items():
    band = residual_band(*STATES[state])
    for (quantity, value), pw, display in zip(results(nets[state]).items(), pw_values, DISPLAY):
        source = "Buses, Gen MW/Mvar" if quantity.startswith("Slack") else "Buses, Volt (kV) / Angle (Deg)"
        rows.append([state, quantity, pw, value, max(display, band[quantity]), source])

compare(rows)

,Case,Quantity,PowerWorld,pandapower,Tolerance,Source,Abs error
0,"Bypassed, 2200 MW",Bus B voltage [kV],765.010,765.011,0.0803,"Buses, Volt (kV) / Angle (Deg)",0.0011
1,"Bypassed, 2200 MW",Bus B angle [deg],-21.370,-21.368,0.0060,"Buses, Volt (kV) / Angle (Deg)",0.0022
2,"Bypassed, 2200 MW",Three voltage [kV],765.000,765.000,0.0005,"Buses, Volt (kV) / Angle (Deg)",0.0000
3,"Bypassed, 2200 MW",Three angle [deg],-0.000,-0.003,0.0050,"Buses, Volt (kV) / Angle (Deg)",0.0025
4,"Bypassed, 2200 MW",Four voltage [kV],765.010,765.011,0.0803,"Buses, Volt (kV) / Angle (Deg)",0.0011
5,"Bypassed, 2200 MW",Four angle [deg],-21.370,-21.365,0.0060,"Buses, Volt (kV) / Angle (Deg)",0.0048
6,"Bypassed, 2200 MW",Slack P [MW],2200.000,2200.000,0.5000,"Buses, Gen MW/Mvar",0.0000
7,"Bypassed, 2200 MW",Slack Q [Mvar],-0.160,-0.163,0.6808,"Buses, Gen MW/Mvar",0.0027
8,"In service, 2200 MW",Bus B voltage [kV],777.175,777.176,0.0507,"Buses, Volt (kV) / Angle (Deg)",0.0013
9,"In service, 2200 MW",Bus B angle [deg],-14.920,-14.920,0.0050,"Buses, Volt (kV) / Angle (Deg)",0.0002


In [4]:
# 4. What the capacitors buy
# The line itself (3 -> 4) still needs about the same angle to carry the power; the capacitors push the
# voltage at Three ahead of the slack and pull Four back toward Bus B, so the angle the system sees end to end
# (1 -> 2) shrinks. That end-to-end angle is what sets the steady-state stability margin.
rows = []
for state, net in nets.items():
    va = net.res_bus.va_degree
    rows.append({"State": state, "Load [MW]": STATES[state][0],
                 "Angle across the line 3-4 [deg]": va.iat[3] - va.iat[2],
                 "Angle end to end 1-2 [deg]": va.iat[1] - va.iat[0],
                 "Bus B voltage [pu]": net.res_bus.vm_pu.iat[1],
                 "Series X end to end [ohm]": X_OHM + (2 * X_CAP * Z_BASE if STATES[state][1] else 0.0)})
gain = pd.DataFrame(rows)

# 3000 MW with the capacitors in service sits at the same end-to-end angle as 2200 MW without them,
# i.e. the same stability margin for 36 % more power
angle = gain.set_index("State")["Angle end to end 1-2 [deg]"]
assert abs(angle["In service, 3000 MW"] - angle["Bypassed, 2200 MW"]) < 0.05
gain.round(3)

,State,Load [MW],Angle across the line 3-4 [deg],Angle end to end 1-2 [deg],Bus B voltage [pu],Series X end to end [ohm]
0,"Bypassed, 2200 MW",2200.0,-21.363,-21.368,1.000,96.902
1,"In service, 2200 MW",2200.0,-21.089,-14.920,1.016,67.874
2,"In service, 3000 MW",3000.0,-30.022,-21.364,0.979,67.874


In [5]:
# 5. Three ways to model "Bypassed"
# A zero-impedance switch is the textbook short circuit and is off by under 0.01 deg, which no study would
# notice; it also leaves Three at exactly 0.00 deg, while PowerWorld shows -0.00 because its bypass branch
# still carries a sliver of reactance. Mapping the flag to in_service=False is the real trap: PowerWorld keeps
# the branch Closed, but pandapower then opens the circuit, the run still "converges" and Bus B is cut off.
def bypass_variant(kind):
    net = build_case(2200.0, caps_in=False)
    if kind == "zero-impedance switch":
        for i, row in net.impedance.iterrows():
            pp.create_switch(net, row.from_bus, row.to_bus, et="b", closed=True)
        net.impedance["in_service"] = False
    elif kind == "impedance in_service=False":
        net.impedance["in_service"] = False
    pp.runpp(net, numba=False)
    return net


rows = []
for kind in [f"low-impedance branch, X = {X_BYPASS} pu", "zero-impedance switch", "impedance in_service=False"]:
    net = bypass_variant(kind)
    rows.append({"Bypass modeled as": kind, "Converged": net.converged,
                 "Unsupplied buses": sorted(top.unsupplied_buses(net)),
                 "Bus B voltage [kV]": net.res_bus.vm_pu.iat[1] * V_KV,
                 "Bus B angle [deg]": net.res_bus.va_degree.iat[1],
                 "Three angle [deg]": net.res_bus.va_degree.iat[2],
                 "Slack Q [Mvar]": net.res_ext_grid.q_mvar.iat[0],
                 "Load served [MW]": net.res_load.p_mw.iat[0]})
rows.append({"Bypass modeled as": "PowerWorld", "Converged": True, "Unsupplied buses": [],
             "Bus B voltage [kV]": 765.010, "Bus B angle [deg]": -21.37, "Three angle [deg]": -0.00,
             "Slack Q [Mvar]": -0.16, "Load served [MW]": 2200.0})
pd.DataFrame(rows).round(4)

,Bypass modeled as,Converged,Unsupplied buses,Bus B voltage [kV],Bus B angle [deg],Three angle [deg],Slack Q [Mvar],Load served [MW]
0,"low-impedance branch, X = 1e-05 pu",True,[],765.0111,-21.3678,-0.0025,-0.1627,2200.0
1,zero-impedance switch,True,[],765.0266,-21.3623,0.0000,-0.3912,2200.0
2,impedance in_service=False,True,"[1, 2, 3]",NaN,NaN,NaN,0.0000,0.0
3,PowerWorld,True,[],765.0100,-21.3700,-0.0000,-0.1600,2200.0
